# RETFound — Semi-supervised và Few-shot trên Colab

Notebook này nạp `checkpoint-best.pth` từ Google Drive, dùng split `train`/`val` cố định và giữ kín `test`. Chọn GPU runtime trước khi chạy.

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Hãy chọn Runtime > Change runtime type > T4 GPU')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Clone source và cài dependencies

Nếu repository private, hãy clone bằng cơ chế xác thực riêng của bạn; không ghi token trực tiếp vào notebook.

In [ ]:
import os, subprocess, sys
from pathlib import Path

GITHUB_USERNAME = 'Bang334'
GITHUB_REPO = 'dr-diagnostic-system'
GITHUB_BRANCH = 'feat/merged-dataset-training'
REPO_DIR = Path('/content') / GITHUB_REPO
REPO_URL = f'https://github.com/{GITHUB_USERNAME}/{GITHUB_REPO}.git'

if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', GITHUB_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', GITHUB_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', GITHUB_BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '-b', GITHUB_BRANCH, REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'ai/grading/requirements-train.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'ai/semi_supervised/requirements-research.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'kaggle>=2.2.2'], check=True)
print(f'Đã sẵn sàng tại {REPO_DIR}')

## Chọn phương pháp, checkpoint và dữ liệu

- **Checkpoint:** file `checkpoint-best.pth` từ run supervised trên Drive.
- **Dataset có nhãn:** mặc định tự tải bộ fundus gộp từ Kaggle giống notebook grading; cũng có thể nhập thư mục/ZIP.
- **Unlabeled source:** chỉ cần cho semi-supervised; nhập `kaggle:owner/dataset`, thư mục hoặc ZIP ảnh ngoài chưa có nhãn. Không chọn bất kỳ split nào của dataset có nhãn.

Khi dùng `kaggle:...`, tạo Colab Secret tên `KAGGLE_API_TOKEN`. Không ghi token trực tiếp vào notebook.

In [ ]:
import ipywidgets as widgets
from datetime import datetime
from IPython.display import display

DRIVE_ROOT = Path('/content/drive/MyDrive')
checkpoint_candidates, zip_candidates, named_dirs = [], [], []
for current_root, directories, files in os.walk(DRIVE_ROOT):
    current = Path(current_root)
    if 'checkpoint-best.pth' in files:
        checkpoint_candidates.append(str(current / 'checkpoint-best.pth'))
    zip_candidates.extend(str(current / name) for name in files if name.lower().endswith('.zip'))
    named_dirs.extend(
        str(current / name) for name in directories
        if any(key in name.lower() for key in ('fundus', 'dataset', 'unlabeled'))
    )
checkpoint_candidates.sort()
zip_candidates.sort()
named_dirs = sorted(set(named_dirs))

DEFAULT_LABELED_SOURCE = 'kaggle:sehastrajits/fundus-aptosddridirdeyepacsmessidor'
method_widget = widgets.ToggleButtons(
    options=[('Semi-supervised', 'semi'), ('Few-shot', 'fewshot')],
    description='Phương pháp:'
)
checkpoint_widget = widgets.Combobox(
    options=checkpoint_candidates, value=checkpoint_candidates[0] if checkpoint_candidates else '',
    placeholder='/content/drive/MyDrive/.../checkpoint-best.pth',
    description='Checkpoint:', ensure_option=False, layout=widgets.Layout(width='95%')
)
dataset_widget = widgets.Combobox(
    options=[DEFAULT_LABELED_SOURCE] + named_dirs + zip_candidates, value=DEFAULT_LABELED_SOURCE,
    placeholder='kaggle:owner/dataset hoặc thư mục/ZIP có split',
    description='Dataset:', ensure_option=False, layout=widgets.Layout(width='95%')
)
unlabeled_widget = widgets.Combobox(
    options=named_dirs + zip_candidates, placeholder='kaggle:owner/dataset hoặc thư mục/ZIP ảnh chưa nhãn',
    description='Unlabeled:', ensure_option=False, layout=widgets.Layout(width='95%')
)
run_name_widget = widgets.Text(
    value=f"retfound_research_{datetime.now():%Y%m%d_%H%M}", description='Tên run:', layout=widgets.Layout(width='70%')
)
display(method_widget, checkpoint_widget, dataset_widget, unlabeled_widget, run_name_widget)
print(f'Tìm thấy {len(checkpoint_candidates)} best checkpoint trên Drive.')

In [ ]:
import getpass, shutil, zipfile
from google.colab import userdata
from ai.grading.train import find_predefined_splits
from ai.semi_supervised.research_utils import discover_images, load_grading_checkpoint

def get_kaggle_token():
    try:
        token = userdata.get('KAGGLE_API_TOKEN')
    except Exception:
        token = getpass.getpass('Dán Kaggle API token: ').strip()
    if not token:
        raise RuntimeError('Chưa cung cấp KAGGLE_API_TOKEN')
    os.environ['KAGGLE_API_TOKEN'] = token

def download_kaggle_source(dataset_ref, extract_name):
    target = Path('/content/selected_data') / extract_name
    marker = target / '.kaggle_source'
    if marker.is_file() and marker.read_text(encoding='utf-8').strip() == dataset_ref:
        print(f'Dùng lại dataset Kaggle đã tải: {target}')
        return target.resolve()
    if target.exists():
        shutil.rmtree(target)
    target.mkdir(parents=True)
    get_kaggle_token()
    kaggle_cli = [sys.executable, '-m', 'kaggle']
    print(f'Kiểm tra quyền Kaggle: {dataset_ref}')
    subprocess.run(
        kaggle_cli + ['datasets', 'files', '-d', dataset_ref, '--page-size', '20'],
        check=True, capture_output=True, text=True,
    )
    print(f'Đang tải Kaggle dataset {dataset_ref}...')
    subprocess.run(
        kaggle_cli + ['datasets', 'download', '-d', dataset_ref, '-p', str(target)],
        check=True,
    )
    archives = sorted(target.glob('*.zip'))
    if not archives:
        raise FileNotFoundError(f'Kaggle không tạo ZIP trong {target}')
    for archive_path in archives:
        with zipfile.ZipFile(archive_path) as archive:
            archive.extractall(target)
        archive_path.unlink()
    marker.write_text(dataset_ref, encoding='utf-8')
    return target.resolve()

def resolve_source(raw_value, extract_name):
    raw_value = raw_value.strip()
    if not raw_value:
        raise ValueError(f'Chưa chọn nguồn dữ liệu cho {extract_name}')
    if raw_value.lower().startswith('kaggle:'):
        dataset_ref = raw_value.split(':', 1)[1].strip()
        if '/' not in dataset_ref:
            raise ValueError('Kaggle dataset phải có dạng kaggle:owner/dataset')
        return download_kaggle_source(dataset_ref, extract_name)
    source = Path(raw_value).expanduser()
    if not source.exists():
        raise FileNotFoundError(f'Không tìm thấy: {source}')
    if source.is_dir():
        return source.resolve()
    if source.suffix.lower() != '.zip':
        raise ValueError(f'Chỉ chấp nhận thư mục hoặc ZIP: {source}')
    target = Path('/content/selected_data') / extract_name
    if target.exists():
        shutil.rmtree(target)
    target.mkdir(parents=True)
    with zipfile.ZipFile(source) as archive:
        archive.extractall(target)
    return target.resolve()

METHOD = method_widget.value
CHECKPOINT_PATH = Path(checkpoint_widget.value).expanduser().resolve()
if not CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(f'Checkpoint không tồn tại: {CHECKPOINT_PATH}')
# Kiểm tra metadata/kiến trúc trước khi bắt đầu tác vụ dài.
bundle = load_grading_checkpoint(CHECKPOINT_PATH, torch.device('cpu'), require_ce=True)
del bundle
import gc
gc.collect()

dataset_source = resolve_source(dataset_widget.value, 'labeled_dataset')
split_dirs = find_predefined_splits(dataset_source)
DATASET_DIR = next(iter(split_dirs.values())).parent.resolve()

UNLABELED_DIR = None
if METHOD == 'semi':
    if unlabeled_widget.value.strip() == dataset_widget.value.strip():
        raise ValueError('Dataset có nhãn và nguồn unlabeled không được giống nhau')
    UNLABELED_DIR = resolve_source(unlabeled_widget.value, 'unlabeled_dataset')
    print(f'Ảnh chưa nhãn tìm thấy: {len(discover_images(UNLABELED_DIR)):,}')

OUTPUT_DIR = DRIVE_ROOT / 'retfound_research' / run_name_widget.value / METHOD
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Phương pháp : {METHOD}')
print(f'Checkpoint  : {CHECKPOINT_PATH}')
print(f'Dataset     : {DATASET_DIR}')
print(f'Unlabeled   : {UNLABELED_DIR}')
print(f'Output      : {OUTPUT_DIR}')
print('Test split được giữ kín và không dùng để chọn model.')

## Chạy thí nghiệm

Thông số mặc định thận trọng cho RETFound trên T4. Semi-supervised giữ pseudo-label từ confidence 0.95; few-shot là 5-shot, mở một transformer block cuối.

In [ ]:
if METHOD == 'semi':
    cmd = [
        sys.executable, '-m', 'ai.semi_supervised.semi_supervised_training',
        '--checkpoint', str(CHECKPOINT_PATH),
        '--dataset-dir', str(DATASET_DIR),
        '--unlabeled-dir', str(UNLABELED_DIR),
        '--output-dir', str(OUTPUT_DIR),
        '--epochs', '6', '--patience', '3',
        '--batch-size', '2', '--accum-steps', '8',
        '--head-lr', '1e-5', '--backbone-lr', '1e-6', '--min-lr', '1e-7',
        '--threshold', '0.95', '--pseudo-weight', '0.25',
        '--max-pseudo-per-class', '2000', '--seed', '42',
    ]
else:
    cmd = [
        sys.executable, '-m', 'ai.semi_supervised.few_shot_demo',
        '--checkpoint', str(CHECKPOINT_PATH),
        '--dataset-dir', str(DATASET_DIR),
        '--output-dir', str(OUTPUT_DIR),
        '--epochs', '8', '--train-episodes', '40', '--val-episodes', '20',
        '--shots', '5', '--queries', '3',
        '--unfreeze-last-blocks', '1', '--forward-batch-size', '2',
        '--encoder-lr', '1e-6', '--projection-lr', '1e-4',
        '--patience', '3', '--seed', '42',
    ]
print(' '.join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
import json
summary_path = OUTPUT_DIR / 'summary.json'
if not summary_path.is_file():
    raise FileNotFoundError(f'Run chưa tạo summary: {summary_path}')
summary = json.loads(summary_path.read_text(encoding='utf-8'))
print(json.dumps(summary, indent=2, ensure_ascii=False))
if summary.get('test_split_used') is not False:
    raise RuntimeError('Safety check failed: test split usage was not recorded as false')
if METHOD == 'semi':
    import pandas as pd
    display(pd.read_csv(OUTPUT_DIR / 'pseudo_labels.csv').head(20))

## Sau khi chạy

Không đánh giá test cho từng lần chỉnh tham số. Chỉ sau khi khóa phương pháp bằng validation mới dùng script đánh giá độc lập trên test. ProtoNet cần support set khi inference và không được chép trực tiếp vào service grading.